In [ ]:
print("Hello, World!")

In [1]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 32.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 26.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pandas numpy matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 28.0 MB/s  0:00:00 eta 0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 22.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 30.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install Pool

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for Pool: filename=pool-0.1.2.dev0-py3-none-any.whl size=23175 sha256=d537bbb027e7eaf586d007ae2e16d4173f43c13a8596e45313ad4d7a03544685
  Stored in directory: /Users/abhisheksarkar/Library/Caches/pip/wheels/ca/7a/9e/55d75c5d1effae7385b53ac7fd2ff94672c54e79e29a9beaf6
Successfully built Pool
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install multiprocess

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [multiprocess]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
Generate quickkart_orders.csv — 100,000 synthetic QuickKart order records.

Schema matches the QuickKart HDFS/MapReduce teaching notebooks:
    OrderID, Region, Category, OrderValue, Timestamp

Usage:
    python generate_quickkart_orders.py
"""

import pandas as pd
import numpy as np

np.random.seed(42)

N_ORDERS = 300000

categories = [
    "Fruits & Vegetables", "Dairy", "Snacks", "Beverages",
    "Household", "Personal Care", "Bakery", "Frozen Foods",
]

regions = ["North", "South", "West"]

df = pd.DataFrame({
    "OrderID": range(1, N_ORDERS + 1),
    "Region": np.random.choice(regions, N_ORDERS, p=[0.4, 0.35, 0.25]),
    "Category": np.random.choice(categories, N_ORDERS),
    "OrderValue": np.round(np.random.gamma(shape=3.5, scale=120, size=N_ORDERS), 2),
    # 300,000 orders at ~1000/day starting Jan 1, 2026 -- spans ~10 months
    "Timestamp": pd.date_range("2026-01-01", periods=N_ORDERS, freq="86400ms"),
})

df.to_csv("quickkart_orders.csv", index=False)

print(f"quickkart_orders.csv written with {len(df):,} rows")
print(df.head())
print(f"\nDate range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
print(f"File size: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")


In [1]:
import pandas as pd, os, shutil

# Load QuickKart orders
df = pd.read_csv("/Users/abhisheksarkar/IdeaProjects/NMIMS/.ipynb_checkpoints/quickkart_orders.csv")

# 1. Split into "blocks"
block_size = len(df) // 3
blocks = [df.iloc[i*block_size:(i+1)*block_size] for i in range(3)]

# 2. Create "DataNode" folders, write each block with replication factor 2
os.makedirs("cluster/datanode1", exist_ok=True)
os.makedirs("cluster/datanode2", exist_ok=True)
os.makedirs("cluster/datanode3", exist_ok=True)

blocks[0].to_csv("cluster/datanode1/block1.csv", index=False)
blocks[0].to_csv("cluster/datanode2/block1.csv", index=False)  # replica
blocks[1].to_csv("cluster/datanode2/block2.csv", index=False)
blocks[1].to_csv("cluster/datanode3/block2.csv", index=False)  # replica
blocks[2].to_csv("cluster/datanode3/block3.csv", index=False)
blocks[2].to_csv("cluster/datanode1/block3.csv", index=False)  # replica

# 3. "NameNode" metadata
metadata = {
    "block1.csv": ["datanode1", "datanode2"],
    "block2.csv": ["datanode2", "datanode3"],
    "block3.csv": ["datanode3", "datanode1"],
}
print(metadata)

{'block1.csv': ['datanode1', 'datanode2'], 'block2.csv': ['datanode2', 'datanode3'], 'block3.csv': ['datanode3', 'datanode1']}


In [2]:
os.remove("cluster/datanode1/block1.csv")  # simulate DataNode1 failure

# NameNode logic: try first replica, fall back to second
def read_block(block, metadata):
    for node in metadata[block]:
        path = f"cluster/{node}/{block}"
        if os.path.exists(path):
            print(f"Read {block} from {node}")
            return pd.read_csv(path)
    raise FileNotFoundError("All replicas lost!")

read_block("block1.csv", metadata)

Read block1.csv from datanode2


,OrderID,Region,Category,OrderValue,Timestamp
0,1,North,Snacks,243.59,2026-01-01 00:00:00.000
1,2,West,Snacks,476.99,2026-01-01 00:01:26.400
2,3,South,Personal Care,356.92,2026-01-01 00:02:52.800
3,4,South,Personal Care,294.29,2026-01-01 00:04:19.200
4,5,North,Snacks,409.05,2026-01-01 00:05:45.600
...,...,...,...,...,...
99995,99996,West,Bakery,418.70,2026-04-10 23:52:48.000
99996,99997,West,Personal Care,168.81,2026-04-10 23:54:14.400
99997,99998,South,Fruits & Vegetables,398.24,2026-04-10 23:55:40.800
99998,99999,South,Frozen Foods,472.46,2026-04-10 23:57:07.200


In [3]:
import multiprocessing as mp

# MAP: runs independently per "node" (per block file)
def map_function(filepath):
    chunk = pd.read_csv(filepath)
    return chunk.groupby("Category")["OrderValue"].agg(["count", "sum"])

block_files = [
    "cluster/datanode1/block3.csv",
    "cluster/datanode2/block2.csv",
    "cluster/datanode3/block1.csv" if os.path.exists("cluster/datanode3/block1.csv") else "cluster/datanode2/block1.csv",
]

# Simulate parallel execution across nodes
ctx = mp.get_context("fork")
with ctx.Pool(3) as pool:
    map_results = pool.map(map_function, block_files)

# SHUFFLE + REDUCE: combine partial results by key
combined = pd.concat(map_results)
final_result = combined.groupby(combined.index).sum()
print(final_result)


                     count          sum
Category                               
Bakery               37597  15767794.41
Beverages            37666  15827369.90
Dairy                37388  15705149.89
Frozen Foods         37480  15703459.64
Fruits & Vegetables  37487  15717101.54
Household            37470  15768079.05
Personal Care        37284  15692592.98
Snacks               37628  15804283.73
